In [ ]:
import numpy as np
from scipy.integrate import fixed_quad, quad
import os
import plotly.graph_objects as go
from scipy.special import j0, jv
import pandas as pd
import plotly.graph_objects as go

In [ ]:
# Leitura do arquivo com separação por espaços
data_atlas = pd.read_csv(
    "../../../data/sigma_tot_2/ensemble_StRh_atlas.dat",
    delim_whitespace=True,
    header=None,
    nrows=70  # lê apenas as 70 primeiras linhas
)

x_atlas = data_atlas[0].to_numpy()
y_atlas = data_atlas[1].to_numpy()
y_error_atlas = data_atlas[2].to_numpy()

In [ ]:
# === Global Configuration and Constants ===
start_sqrt_s = 1  # Global parameter controlling energy scale
b_0 = (33 - 6) / (12 * np.pi)  # β0 for nf=3
Lambda = 0.284  # ΛQCD in GeV
gamma_1 = 0.084
gamma_2 = 2.36
rho = 4.0

sigma_tot_lst = []
sqrt_s_lst = []
error_lst = []

s0 = 1.0  # GeV^2

epsilon_atlas = 0.0729

model_params = {
    'atlas': {
        'pl':  {'mg': 0.412, 'a1': 1.652, 'a2': 1.479}
    }
}

epsilon_values = {
    'atlas': epsilon_atlas
}


In [ ]:

# === Auxiliary Functions for Physical Model ===
def m2_pl(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return (mg ** 4 / (q2 + mg ** 2)) * ratio ** (gamma_2 - 1)

def get_m2_function(mass_model):
    return m2_pl

def G_p(q2, a1, a2):
    return np.exp(-(a1 * q2 + a2 * q2 ** 2))

def alpha_D(q2, mg, m2_func):
    m2 = m2_func(q2, mg)
    return 1.0 / (b_0 * (q2 + m2) * np.log((q2 + 4 * m2) / (Lambda ** 2)))

def T_1(k, q, phi, mg, a1, a2, m2_func):
    q2 = q ** 2
    qk_cos = q * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2

    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    G0 = G_p(q2, a1, a2)

    return alpha_D_plus * alpha_D_minus * G0 ** 2

def T_2(k, q, phi, mg, a1, a2, m2_func):
    q2 = q ** 2
    qk_cos = q * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2

    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)

    factor = q2 + 9 * abs(k ** 2 - q2 / 4)

    G0 = G_p(q2, a1, a2)
    G_minus = G_p(factor, a1, a2)

    return alpha_D_plus * alpha_D_minus * G_minus * (2 * G0 - G_minus)

def integrand(y, x, mg, a1, a2, m2_func):
    k = sqrt_s * x
    phi = 2 * np.pi * y
    jacobian = 2 * np.pi * sqrt_s

    return k * (T_1(k, 0.0, phi, mg, a1, a2, m2_func) - T_2(k, 0.0, phi, mg, a1, a2, m2_func)) * jacobian

def amp_calculation(diff_T, s, epsilon):
    alpha_pomeron = 1.0 + epsilon
    regge_factor = (s / s0) ** alpha_pomeron
    
    return 1j * 8.0 * regge_factor * diff_T

def sigma_tot(amp_value, s):
    return amp_value.imag / s * 0.389379323


In [ ]:
amp_born_lst = []

sqrt_s_lst = []

# === Main Function ===
global start_sqrt_s
global sqrt_s

max_sqrt_s = 13000
step = 100
n_points = 10000

# Using only PL model with ATLAS
mass_model = 'pl'
ensemble = 'atlas'

fig = go.Figure()

sigma_tot_lst = []




m2_func = get_m2_function(mass_model)
params = model_params[ensemble][mass_model]
mg, a1, a2 = params['mg'], params['a1'], params['a2']
epsilon = epsilon_values[ensemble]

sqrt_s = start_sqrt_s
while sqrt_s <= max_sqrt_s:
    def inner_integral(x):
        return fixed_quad(
            lambda y: integrand(y, x, mg, a1, a2, m2_func),
            0, 1,
            n=n_points
        )[0]

    integral_value = fixed_quad(
        inner_integral,
        0, 1,
        n=n_points
    )[0]

    diff_T = integral_value
    s = sqrt_s * sqrt_s

    amp_value = amp_calculation(diff_T, s, epsilon)
    sigma_tot_value = sigma_tot(amp_value, s)

    sigma_tot_lst.append(sigma_tot_value)
    sqrt_s_lst.append(sqrt_s)
    amp_born_lst.append(amp_value)

    sqrt_s += step

# Add PL model trace
fig.add_trace(go.Scatter(
    x=sqrt_s_lst,
    y=sigma_tot_lst,
    mode='lines+markers',
    line=dict(
        color='blue',
        width=2
    ),
    marker=dict(
        size=4
    ),
    name='PL Model (ATLAS)'
))

# Add ATLAS data
fig.add_trace(go.Scatter(
    x=x_atlas,
    y=y_atlas,
    mode='markers',
    marker=dict(
        color='black',
        size=6,
        symbol='square'
    ),
    error_y=dict(
        type='data',
        array=y_error_atlas,
        visible=True
    ),
    name='ATLAS Data'
))

# Configure layout
fig.update_layout(
    title='Sigma Tot vs. sqrt(s) - PL Model with ATLAS Data',
    xaxis=dict(
        title='sqrt(s) [GeV]',
        type='log',
    ),
    yaxis=dict(
        title='Sigma Tot [mb]',
    ),
    showlegend=True,
    legend=dict(
        title='Model/Data'
    ),
    plot_bgcolor='white',
    hovermode='x unified'
)

fig.update_xaxes(gridcolor='lightgray')
fig.update_yaxes(gridcolor='lightgray')

# fig.show(renderer="browser")
# fig.write_html("results/sigma_tot/sigma_tot_pl_atlas.html")
# fig.write_image("results/sigma_tot/sigma_tot_pl_atlas.pdf", width=1200, height=600)


In [25]:
print(sigma_tot_lst )

[24.575747812125137, 51.42321435125401, 56.85063273213241, 60.29819839833269, 62.87352336493672, 64.94798662837857, 66.69438187542117, 68.20795704914822, 69.54709213994849, 70.75029097682688, 71.84435407967825, 72.84872370156677, 73.77797432830575, 74.6433266345851, 75.45361239830928, 76.21591391236913, 76.93600176335273, 77.61864300679294, 78.26782336578592, 78.88691081606744, 79.47877824772374, 80.04589694774758, 80.5904088851948, 81.11418333804119, 81.61886177829817, 82.10589383179641, 82.57656636911575, 83.03202725030765, 83.47330486523805, 83.90132433587073, 84.31692104489646, 84.72085200535612, 85.11380547360179, 85.4964091228529, 85.86923702952184, 86.23281567425735, 86.58762912056582, 86.93412350321759, 87.27271093442513, 87.60377291651162, 87.92766333436356, 88.24471108852823, 88.55522241974774, 88.8594829675109, 89.15775959848608, 89.45030203515951, 89.73734431043195, 90.01910607011999, 90.29579374213874, 90.56760158848356, 90.8347126538936, 91.09729962319214, 91.355525597700

In [ ]:
# print(sqrt_s_lst)
sqrt_s_lst.remove(1)


In [ ]:
print(sqrt_s_lst)
print(amp_born_lst)
print(sigma_tot_lst)

In [ ]:

lst_s = []
for key, value in enumerate(sqrt_s_lst):
    lst_s.append(value ** 2)
    # print(f"sqrt(s) = {value:.2f} GeV, s = {lst_s[key]:.2f} GeV^2")

# print(amp_born_lst)

def sigma_tot_eik(s, amp):
    return (4*np.pi)/s * amp.imag * 0.389379323


In [ ]:
from scipy.integrate import quad
import numpy as np
from scipy.special import j0

# Limits
q_upper = 0.1  # inner integral
b_upper = 40 # outer integral

# Inner integral over q
def inner_integral(b, s, amp):
    integrand = lambda q: q * j0(b * q)  # you can later include s, amp if needed
    result, _ = quad(integrand, 0, q_upper, limit=200)
    return result * (1/s) * amp

# Outer integrand
def outer_integrand(b, s, amp):
    inner_result = inner_integral(b, s, amp)
    return 1j * s * b * (1 - np.exp(1j*inner_result)) # amp applied here as example

# Split real and imaginary parts
def outer_real(b, s, amp):
    return np.real(outer_integrand(b, s, amp))

def outer_imag(b, s, amp):
    return np.imag(outer_integrand(b, s, amp))

# Compute double integral for given s, amp
def compute_double_integral(s, amp):
    real_part, _ = quad(outer_real, 0, b_upper, args=(s, amp), limit=500)
    imag_part, _ = quad(outer_imag, 0, b_upper, args=(s, amp), limit=500)
    return real_part + 1j * imag_part

lst_amp_eik = []

# Example loop
for s, amp in zip(lst_s, amp_born_lst):
    amp_eik_val = compute_double_integral(s, amp)
    # print(f"s={s}, amp={amp:.2e}, result={amp_eik_val:.2e}")
    lst_amp_eik.append(amp_eik_val)


In [ ]:

lst_sigma_tot_eik = []

for amp_eik_val, s_val in zip(lst_amp_eik, lst_s):
    sigma_tot_eik_val = sigma_tot_eik(s_val, amp_eik_val)

    # print(f's = {s_val}, amp eik = {amp_eik_val}, sigma tot eik = {sigma_tot_eik_val}')
    
    lst_sigma_tot_eik.append(sigma_tot_eik_val)

In [ ]:
lst_imag_val_amp_eik = []
for values in lst_amp_eik:
    img_part = values.imag
    lst_imag_val_amp_eik.append(img_part)


In [ ]:
def create_iterative_graph(fig, x_data, y_data, title:str, x_axis_name:str, y_axis_name:str):

    fig.add_trace(go.Scatter(
    x = x_data,
    y = y_data,
    mode='lines+markers')
)

    fig.update_layout(
        title=title,
        xaxis=dict(
            title = x_axis_name,
            type='log',
        ),
        yaxis=dict(
            title= y_axis_name,
            # range=[80, 120]
        ),
        showlegend=False,
        plot_bgcolor='white',
        hovermode='x unified'
    )
        
    fig.update_xaxes(gridcolor='lightgray')
    fig.update_yaxes(gridcolor='lightgray')



In [ ]:
fig_amp_eik = go.Figure()

create_iterative_graph(fig_amp_eik, sqrt_s_lst, lst_imag_val_amp_eik, 'Amp eikonal vs sqrt', 'sqrt s [GeV]', 'Im(Amp_eikonal)')

# os.makedirs("../../../results/amp_eikonal", exist_ok=True)
# file_name = 'amp_eikonal.pdf'
# save_path = os.path.join("../../../results/amp_eikonal", file_name)

fig_amp_eik.show(renderer = 'browser')

# fig.write_image(save_path, width=1200, height=600)

In [ ]:
# fig_sigma_eik = go.Figure()

# create_iterative_graph(fig_sigma_eik, sqrt_s_lst, lst_sigma_tot_eik, 'sigma eik vs sqrt', 'sqrt s [GeV]', 'sigma eik [mb]')

# # os.makedirs("../../../results/amp_eikonal", exist_ok=True)
# # file_name = 'amp_eikonal.pdf'
# # save_path = os.path.join("../../../results/amp_eikonal", file_name)

# fig_sigma_eik.show(renderer = 'browser')

# # fig.write_image(save_path, width=1200, height=600)